In [1]:
!pip install ramantune


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
from ramantune.pipeline.raman_pipeline import RamanPipeline

from ramantune.search.search_space import DenoiserSpace, BaselineSpace, NormalizerSpace, ClassifierSpace, FeatureSelectionSpace
from ramantune.utils.config import DENOISING_STR, BASELINE_STR, NORMALIZE_STR, FEATURE_SELECTION_STR, CLASSIFIER_STR
from ramantune.search.strategies import GridSearchStrategy
from ramantune.search import RamanSearch

from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("bin/ovarian_small.csv")
groups = df['patient']

y = df['label'].values
X = df.drop(columns=['label', 'patient'])

In [15]:
df

,401.226563,402.488281,403.75,405.010742,406.271484,407.53125,408.791992,410.051758,411.311523,412.570313,...,1995.704102,1996.632813,1997.561523,1998.489258,1999.416992,2000.344727,2001.272461,2002.199219,patient,label
0,45522.917969,45744.746094,46069.648438,45632.875000,45732.406250,45830.031250,45929.640625,45731.687500,45521.972656,45642.976563,...,14715.435547,14464.926758,14700.097656,14607.922852,14354.601563,14507.996094,14600.679688,14381.378906,1,HC
1,29890.923828,30093.746094,30147.046875,30052.671875,30222.613281,29856.013672,30201.003906,29952.884766,30206.605469,29841.615234,...,9622.037109,9497.227539,9615.257813,9485.115234,9450.002930,9483.558594,9506.558594,9474.852539,1,HC
2,15597.038086,15347.965820,15481.545898,15574.374023,15424.262695,15635.716797,15518.610352,15488.981445,15511.868164,15672.928711,...,4671.147949,4738.055664,4628.102051,4584.120605,4619.355957,4617.621582,4597.390625,4703.078125,1,HC
3,15505.758789,15616.035156,15776.876953,15702.637695,15579.769531,15760.148438,15436.933594,15751.572266,15825.101563,15900.608398,...,4752.958984,4587.599121,4686.184082,4639.573730,4518.992188,4585.921387,4689.867188,4607.959473,1,HC
4,14361.859375,14415.548828,14651.900391,14433.600586,14605.912109,14379.726563,14361.520508,14586.446289,14406.795898,14534.530273,...,4259.453613,4043.844482,4205.685547,4203.871094,4281.289551,4142.122559,4148.220703,4243.338867,1,HC
5,11835.181641,11863.058594,11655.847656,12066.543945,11767.925781,12044.672852,11761.472656,11806.871094,11733.610352,11790.697266,...,3425.508545,3515.927490,3403.093750,3422.246582,3351.608154,3471.141113,3411.052734,3474.464844,1,HC
6,16016.532227,16288.152344,16091.637695,16161.279297,16215.400391,16238.436523,16096.182617,16224.236328,16089.696289,16038.773438,...,5159.375488,5302.926758,5127.081543,5167.698242,5118.531250,5119.536621,5228.871582,5136.395508,1,HC
7,20543.576172,20724.900391,20789.726563,20805.996094,20892.246094,20677.175781,20971.515625,20721.351563,20570.296875,20829.777344,...,6368.067871,6403.633789,6093.359375,6303.165527,6201.399414,6176.200684,6121.927734,6140.423340,2,HC
8,22912.943359,22677.148438,22699.662109,22735.787109,22909.939453,23123.050781,22836.470703,22736.494141,22794.060547,23028.736328,...,6732.259277,6804.850586,6742.825195,6796.961914,6742.833008,6662.266113,6724.344727,6822.105469,2,HC
9,28399.386719,28289.130859,28310.945313,28530.994141,28416.798828,28380.322266,28326.326172,28243.125000,27710.464844,28286.775391,...,8906.850586,8892.762695,8857.547852,8877.772461,8736.894531,8714.835938,8745.611328,8785.244141,2,HC


In [16]:
df['label'].value_counts()

label
HC    21
OC    21
Name: count, dtype: int64

In [17]:
df['patient'].value_counts()

patient
1     7
2     7
3     7
29    7
30    7
31    7
Name: count, dtype: int64

In [6]:
from ramantune.utils import register_algorithm
from ramantune.custom import RamanPipelineStep
from orpl.baseline_removal import bubblefill

@register_algorithm(category=NORMALIZE_STR, name="snv")
class SNVNormalization(RamanPipelineStep):
    def __init__(self):
        super().__init__(self.snv_normalization)

    @staticmethod
    def snv_normalization(spectral_data, spectral_axis):
        mean, std = spectral_data.mean(), spectral_data.std()
        return (spectral_data - mean) / std, spectral_axis

@register_algorithm(category=BASELINE_STR, name="bubblefill")
class BubbleFill(RamanPipelineStep):
    def __init__(self, *, min_bubble_widths=50, fit_order=1):
        super().__init__(
            self._bubblefill_call,
            min_bubble_widths=min_bubble_widths,
            fit_order=fit_order
        )

    @staticmethod
    def _bubblefill_call(spectral_data, spectral_axis, *args, **kwargs):
        raman, bubblefill_b = bubblefill(
            spectral_data,
            min_bubble_widths=kwargs.get("min_bubble_widths", 50),
            fit_order=kwargs.get("fit_order", 1))
        return raman, spectral_axis

In [9]:
def setup_param_grid():
  denoiser_list = [
        DenoiserSpace("savgol", {"window_length": [7], "polyorder": [3]}),
  ]

  baseline_list = [
      BaselineSpace("asls", {"lam": [100]}),
      BaselineSpace("bubblefill", {"min_bubble_widths": [50]}),
  ]

  normalization_list = [
      NormalizerSpace("snv"),
      NormalizerSpace("vector"),
  ]

  feature_selection_list = [
      FeatureSelectionSpace(None), # No feature selection
      FeatureSelectionSpace(PCA(),{"n_components": [0.90, 10]}),
  ]

  classifier_list = [
      ClassifierSpace(SVC(),{"C": [0.1, 10], "kernel": ["rbf"], "gamma": ["scale"]}),
      ClassifierSpace(RandomForestClassifier(),{"n_estimators": [100, 200]})
  ]

  param_list = {
      DENOISING_STR: denoiser_list,
      BASELINE_STR: baseline_list,
      NORMALIZE_STR: normalization_list,
      FEATURE_SELECTION_STR: feature_selection_list,
      CLASSIFIER_STR: classifier_list
  }

  return param_list

In [10]:
estimator = RamanPipeline()
param_grid = setup_param_grid()

In [11]:
search = RamanSearch(estimator=estimator,
                     research_strategy=GridSearchStrategy(),
                     param_grid=param_grid,
                     cv=StratifiedGroupKFold(n_splits=2, random_state=42, shuffle=True),
                     return_train_score=True,
                     n_jobs=1,
                     verbose=10,
                     refit="accuracy")

res = search.fit(X, y, groups=groups)

Fitting 2 folds for each of 48 candidates, totalling 96 fits
[CV 1/2; 1/48] START baseline__algorithm=asls, baseline__lam=100, classifier__C=0.1, classifier__algorithm=SVC(), denoising__algorithm=savgol, denoising__polyorder=3, denoising__window_length=7, feature__algorithm=None, normalize__algorithm=snv
[CV 1/2; 1/48] END baseline__algorithm=asls, baseline__lam=100, classifier__C=0.1, classifier__algorithm=SVC(), denoising__algorithm=savgol, denoising__polyorder=3, denoising__window_length=7, feature__algorithm=None, normalize__algorithm=snv; accuracy: (train=0.667, test=0.333) f1: (train=0.400, test=0.250) patient_accuracy: (train=0.667, test=0.333) precision: (train=0.333, test=0.167) recall: (train=0.500, test=0.500) sensitivity: (train=0.500, test=0.500) specificity: (train=0.500, test=0.500) total time=   0.2s
[CV 2/2; 1/48] START baseline__algorithm=asls, baseline__lam=100, classifier__C=0.1, classifier__algorithm=SVC(), denoising__algorithm=savgol, denoising__polyorder=3, denoi

In [12]:
print(search.get_best_params())
print(search.get_best_score())

{'baseline__algorithm': 'asls', 'baseline__lam': 100, 'classifier__C': 10, 'classifier__algorithm': SVC(), 'denoising__algorithm': 'savgol', 'denoising__polyorder': 3, 'denoising__window_length': 7, 'feature__algorithm': PCA(), 'feature__n_components': 10, 'normalize__algorithm': 'snv'}
0.6190476190476191


In [13]:
result_cv = search.get_cv_results(file_path=f"result_add_preprocessing.csv",
                          return_split_scores=False,
                          return_combined_params=True,
                          round_values=True)

In [14]:
result_cv

,denoising,baseline,normalize,feature,classifier,mean_fit_time,std_fit_time,mean_score_time,std_score_time,split0_test_accuracy,...,std_train_sensitivity,split0_test_patient_accuracy,split1_test_patient_accuracy,mean_test_patient_accuracy,std_test_patient_accuracy,rank_test_patient_accuracy,split0_train_patient_accuracy,split1_train_patient_accuracy,mean_train_patient_accuracy,std_train_patient_accuracy
0,"savgol(polyorder=3,window_length=7)",asls(lam=100.0),snv,None,SVC(C=0.1),0.0835,0.0233,0.1306,0.0364,0.3333,...,0.0,0.3333,0.3333,0.3333,0.0000,25,0.6667,0.6667,0.6667,0.0
1,"savgol(polyorder=3,window_length=7)",asls(lam=100.0),snv,None,SVC(C=10.0),0.0729,0.0017,0.1075,0.0102,0.5714,...,0.0,0.6667,0.3333,0.5000,0.1667,2,1.0000,1.0000,1.0000,0.0
2,"savgol(polyorder=3,window_length=7)",asls(lam=100.0),snv,None,RandomForestClassifier(n_estimators=100.0),0.1627,0.0272,0.1426,0.0400,0.2857,...,0.0,0.3333,0.3333,0.3333,0.0000,25,1.0000,1.0000,1.0000,0.0
3,"savgol(polyorder=3,window_length=7)",asls(lam=100.0),snv,None,RandomForestClassifier(n_estimators=200.0),0.3318,0.0180,0.1586,0.0211,0.3333,...,0.0,0.3333,0.3333,0.3333,0.0000,25,1.0000,1.0000,1.0000,0.0
4,"savgol(polyorder=3,window_length=7)",asls(lam=100.0),snv,PCA(n_components=0.9),SVC(C=0.1),0.0585,0.0010,0.0840,0.0023,0.3333,...,0.0,0.3333,0.3333,0.3333,0.0000,25,0.6667,0.6667,0.6667,0.0
5,"savgol(polyorder=3,window_length=7)",asls(lam=100.0),snv,PCA(n_components=10.0),SVC(C=0.1),0.1074,0.0291,0.0846,0.0029,0.3333,...,0.0,0.3333,0.3333,0.3333,0.0000,25,0.6667,0.6667,0.6667,0.0
6,"savgol(polyorder=3,window_length=7)",asls(lam=100.0),snv,PCA(n_components=0.9),SVC(C=10.0),0.0804,0.0228,0.1189,0.0234,0.6667,...,0.0,0.6667,0.3333,0.5000,0.1667,2,1.0000,1.0000,1.0000,0.0
7,"savgol(polyorder=3,window_length=7)",asls(lam=100.0),snv,PCA(n_components=10.0),SVC(C=10.0),0.1077,0.0041,0.1492,0.0551,0.7143,...,0.0,0.6667,0.6667,0.6667,0.0000,1,1.0000,1.0000,1.0000,0.0
8,"savgol(polyorder=3,window_length=7)",asls(lam=100.0),snv,PCA(n_components=0.9),RandomForestClassifier(n_estimators=100.0),0.3302,0.1026,0.2410,0.0774,0.6190,...,0.0,0.6667,0.3333,0.5000,0.1667,2,1.0000,1.0000,1.0000,0.0
9,"savgol(polyorder=3,window_length=7)",asls(lam=100.0),snv,PCA(n_components=10.0),RandomForestClassifier(n_estimators=100.0),0.4399,0.1187,0.3375,0.1031,0.6190,...,0.0,0.6667,0.3333,0.5000,0.1667,2,1.0000,1.0000,1.0000,0.0
